## Claim Extractor

In [1]:
"""
CASM — Claim Extractor (Hybrid Ensemble)
=========================================
Stage 1 of the CASM (Clinical Automated Safety Monitor) pipeline.

Single responsibility: convert a raw LLM clinical response into a list of
structured, typed, and entity-tagged atomic claims.

Dual-pipeline architecture:
  - Med7 (en_core_med7_trf)  → DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION
  - scispaCy (en_core_sci_md) → DISEASE, fallback conditions

Patient context and population flags are handled downstream by the
Knowledge Verifier — not here.

Install:
    pip install spacy scispacy torch
    pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_md-0.5.4.tar.gz
    pip install https://huggingface.co/kormilitzin/en_core_med7_trf/resolve/main/en_core_med7_trf-3.4.2.1-py3-none-any.whl

NumPy compatibility note:
    spaCy/thinc require NumPy < 2.0. If you hit a binary incompatibility error:
        pip install "numpy==1.26" --reinstall
        pip install spacy --reinstall --no-cache
"""

import re
import spacy
import torch
from dataclasses import dataclass
from enum import Enum
from typing import Optional


# ─── Med7 clinical labels ─────────────────────────────────────────────────────

# Entity labels recognised by Med7 as drug/medication names.
MED7_DRUG_LABELS    = {"DRUG"}

# Entity labels recognised by Med7 as dose or strength values (e.g. "200mg", "0.1%").
MED7_DOSAGE_LABELS  = {"DOSAGE", "STRENGTH"}

# Real clinical information from Med7 that is intentionally excluded from
# Claim.entities to keep the entity list focused on named clinical objects.
MED7_CLINICAL_NOISE = {"FREQUENCY", "DURATION", "FORM", "ROUTE"}


# ─── Enums ────────────────────────────────────────────────────────────────────

class ClaimType(str, Enum):
    """
    Classifies the medical nature of an extracted claim.

    Values
    ------
    DOSAGE_CLAIM      : Claim about drug dose, frequency, or administration route.
    DRUG_SAFETY_CLAIM : Claim about drug hazards, contraindications, or warnings.
    DRUG_INTERACTION  : Claim about interactions between two or more drugs.
    PROCEDURAL_CLAIM  : Claim about monitoring, lab tests, or follow-up actions.
    DIAGNOSIS_CLAIM   : Claim about a patient diagnosis or clinical presentation.
    POPULATION_CLAIM  : Claim specific to a patient sub-population (e.g. elderly,
                        pregnant, renal-impaired).
    GENERAL_MEDICAL   : Fallback for claims with medical content that does not
                        fit a more specific category.
    """
    DOSAGE_CLAIM      = "DOSAGE_CLAIM"
    DRUG_SAFETY_CLAIM = "DRUG_SAFETY_CLAIM"
    DRUG_INTERACTION  = "DRUG_INTERACTION"
    PROCEDURAL_CLAIM  = "PROCEDURAL_CLAIM"
    DIAGNOSIS_CLAIM   = "DIAGNOSIS_CLAIM"
    POPULATION_CLAIM  = "POPULATION_CLAIM"
    GENERAL_MEDICAL   = "GENERAL_MEDICAL"


# ─── Data Classes ─────────────────────────────────────────────────────────────

@dataclass
class Entity:
    """
    A single named entity recognised within a claim sentence.

    Attributes
    ----------
    text  : The surface form of the entity as it appears in the sentence.
    label : The NER label assigned by the pipeline that detected it.
              Med7 labels  : DRUG | DOSAGE | STRENGTH | FORM | FREQUENCY | ROUTE | DURATION
              scispaCy labels: DISEASE | CONDITION | DISORDER
    start : Character offset (inclusive) of the entity within the sentence.
    end   : Character offset (exclusive) of the entity within the sentence.
    """
    text:  str
    label: str
    start: int
    end:   int


@dataclass
class Claim:
    """
    A single structured, atomic medical claim extracted from an LLM response.

    Attributes
    ----------
    claim_id              : Unique identifier for this claim (e.g. ``"claim_0"``).
    claim_text            : The raw sentence text from which the claim was derived.
    claim_type            : Semantic category of the claim (see :class:`ClaimType`).
    entities              : All named entities detected by the hybrid pipeline.
    drug_names            : Deduplicated list of drug/medication names found in the claim.
    dosages               : Deduplicated list of dose/strength values (e.g. ``"200mg"``).
    conditions            : Deduplicated list of diseases or clinical conditions mentioned.
    requires_verification : ``True`` when the claim type demands downstream fact-checking
                            (DOSAGE_CLAIM, DRUG_SAFETY_CLAIM, DRUG_INTERACTION, or
                            POPULATION_CLAIM).
    confidence            : Placeholder score in [0, 1]; populated by the Confidence
                            Calibrator stage — always ``0.0`` at extraction time.
    """
    claim_id:              str
    claim_text:            str
    claim_type:            ClaimType
    entities:              list[Entity]
    drug_names:            list[str]
    dosages:               list[str]
    conditions:            list[str]
    requires_verification: bool
    confidence:            float


# ─── Keyword Maps ─────────────────────────────────────────────────────────────

# Maps each ClaimType to a list of trigger keywords used for rule-based
# classification.  The type whose keywords score the most matches in a
# sentence wins; ties default to GENERAL_MEDICAL.
CLAIM_TYPE_KEYWORDS: dict[ClaimType, list[str]] = {
    ClaimType.DOSAGE_CLAIM: [
        "mg", "mcg", "ml", "%", "dose", "dosage", "twice", "once", "three times",
        "daily", "bid", "tid", "qid", "weekly", "monthly", "prescribe",
        "administer", "give", "start", "initiate", "tablet", "capsule", "drops",
    ],
    ClaimType.DRUG_SAFETY_CLAIM: [
        "avoid", "contraindicated", "do not use", "caution", "warning",
        "unsafe", "dangerous", "harmful", "risk", "adverse", "side effect",
        "toxicity", "nephrotoxic", "hepatotoxic", "cardiotoxic",
    ],
    ClaimType.DRUG_INTERACTION: [
        "interaction", "interacts", "combined with", "together with",
        "concurrent", "concomitant", "potentiates", "inhibits", "induces",
        "bleeding risk", "increased risk when",
    ],
    ClaimType.PROCEDURAL_CLAIM: [
        "monitor", "check", "test", "measure", "assess", "every", "weeks",
        "months", "follow up", "review", "screen", "scan", "blood test",
        "lab", "hba1c", "creatinine", "egfr", "blood pressure",
    ],
    ClaimType.DIAGNOSIS_CLAIM: [
        "diagnosis", "diagnose", "patient has", "presents with", "suffering from",
        "consistent with", "indicative of", "suggests", "confirms",
    ],
    ClaimType.POPULATION_CLAIM: [
        "elderly", "pediatric", "children", "pregnant", "renal", "kidney",
        "hepatic", "liver", "geriatric", "neonatal", "trimester",
    ],
}

# Regex patterns for conjunctive phrases that often join two independent
# clinical instructions within a single sentence.  Matching these allows
# compound sentences to be split into separate atomic claims.
SPLIT_CONJUNCTIONS = [
    r"\band also\b", r"\badditionally\b", r"\bfurthermore\b",
    r"\bin addition\b", r"\bmoreover\b", r"\bhowever\b", r"\balternatively\b",
]


# ─── Extractor ────────────────────────────────────────────────────────────────

class ClaimExtractor:
    """
    Hybrid Ensemble Claim Extractor.

    Converts a raw LLM clinical response into a list of structured
    :class:`Claim` objects by running two complementary NLP pipelines:

    Pipeline 1 — Med7 (``en_core_med7_trf``):
        Handles DRUG, DOSAGE, STRENGTH, FORM, FREQUENCY, ROUTE, DURATION.
        High precision for clinical instructions.
        Transformer-based — benefits significantly from GPU acceleration.

    Pipeline 2 — scispaCy (``en_core_sci_md``):
        Fallback for DISEASE/CONDITION detection.
        Only disease-like entities are kept; generic ``ENTITY`` noise is discarded.

    GPU:
        If a CUDA-capable GPU is available, spaCy will automatically use it
        for the Med7 transformer pipeline. ``spacy.prefer_gpu()`` is called
        before any model is loaded (handled in :meth:`__init__`).

    Example usage::

        extractor = ClaimExtractor()
        claims = extractor.extract_as_dict("Prescribe Metformin 500mg twice daily.")
    """

    def __init__(
        self,
        med7_model:   str  = "en_core_med7_trf",
        sci_model:    str  = "en_core_sci_md",
        require_gpu:  bool = False,
    ):
        """
        Initialise the hybrid extractor and load both NLP models.

        Args:
            med7_model  : spaCy model name for the Med7 transformer pipeline.
                          Defaults to ``"en_core_med7_trf"``.
            sci_model   : spaCy model name for the scispaCy pipeline.
                          Defaults to ``"en_core_sci_md"``.
            require_gpu : If ``True``, raises :exc:`RuntimeError` when no
                          CUDA GPU is detected.  Useful in production to
                          prevent silent CPU fallback.  Defaults to ``False``.
        """
        self._setup_device(require_gpu=require_gpu)

        print(f"[ClaimExtractor] Loading Med7 model    : {med7_model}")
        self.nlp_med7 = spacy.load(med7_model)

        print(f"[ClaimExtractor] Loading scispaCy model: {sci_model}")
        self.nlp_sci  = spacy.load(sci_model)

        print("[ClaimExtractor] Hybrid ensemble ready.")

    # ── Device Setup ──────────────────────────────────────────────────────────

    @staticmethod
    def _setup_device(require_gpu: bool = False) -> None:
        """
        Configure spaCy's compute device before any model is loaded.

        Must be called BEFORE ``spacy.load()`` — spaCy allocates model tensors
        at load time and will not migrate them afterwards.

        Priority:
            1. CUDA GPU  (via PyTorch + ``spacy.prefer_gpu()``)
            2. CPU       (silent fallback unless ``require_gpu=True``)

        Note on spaCy GPU=OFF:
            Med7 (``en_core_med7_trf`` 3.4.x) was compiled against spaCy 3.4.
            Running it under spaCy 3.7 means the transformer backend may not
            activate GPU even when CUDA is available — this is a model/version
            mismatch, not a code error. The model still runs correctly on CPU.
            To get full GPU utilisation, retrain Med7 against spaCy 3.7 or
            wait for an official 3.7-compatible release.

        Args:
            require_gpu : If ``True`` and no CUDA device is found, raises
                          :exc:`RuntimeError`.
        """
        if torch.cuda.is_available():
            activated = spacy.prefer_gpu()
            gpu_name  = torch.cuda.get_device_name(0)
            vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1e9
            status    = "ON" if activated else "OFF (model version mismatch — see docstring)"
            print(
                f"[ClaimExtractor] GPU detected : {gpu_name} "
                f"({vram_gb:.1f} GB VRAM) — spaCy GPU={status}"
            )
        else:
            if require_gpu:
                raise RuntimeError(
                    "[ClaimExtractor] require_gpu=True but no CUDA GPU found. "
                    "Install CUDA + torch GPU build, or set require_gpu=False."
                )
            print("[ClaimExtractor] No GPU detected — running on CPU.")

    @staticmethod
    def device_info() -> dict:
        """
        Return a summary of the current compute environment.

        Returns
        -------
        dict
            Keys: ``cuda_available``, ``cuda_device_count``, ``cuda_device_name``,
            ``cuda_vram_gb``, ``spacy_gpu_active``, ``torch_version``,
            ``spacy_version``.
        """
        return {
            "cuda_available":    torch.cuda.is_available(),
            "cuda_device_count": torch.cuda.device_count(),
            "cuda_device_name":  torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
            "cuda_vram_gb":      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2)
                                 if torch.cuda.is_available() else None,
            "spacy_gpu_active":  spacy.prefer_gpu() if torch.cuda.is_available() else False,
            "torch_version":     torch.__version__,
            "spacy_version":     spacy.__version__,
        }

    # ── Public API ────────────────────────────────────────────────────────────

    def extract(self, llm_response: str) -> list[Claim]:
        """
        Extract a list of :class:`Claim` objects from raw LLM output text.

        The text is first split into atomic sentences, then each sentence is
        processed independently through the hybrid NLP pipeline.  Sentences
        with no detectable medical content are silently dropped.

        Args:
            llm_response : Raw text produced by the LLM.

        Returns:
            A list of :class:`Claim` objects, one per extracted sentence.
        """
        sentences = self._split_into_sentences(llm_response)
        claims = []
        for idx, sentence in enumerate(sentences):
            if not sentence.strip():
                continue
            claim = self._process_sentence(sentence, idx)
            if claim:
                claims.append(claim)
        print(f"[ClaimExtractor] {len(claims)} claim(s) extracted.")
        return claims

    def extract_as_dict(self, llm_response: str) -> list[dict]:
        """
        Extract claims and return them as a list of JSON-serialisable dicts.

        Convenience wrapper around :meth:`extract` — delegates all logic there
        and converts the result via :meth:`_to_dict`.

        Args:
            llm_response : Raw text produced by the LLM.

        Returns:
            A list of plain ``dict`` objects mirroring the :class:`Claim` schema.
        """
        return [self._to_dict(c) for c in self.extract(llm_response)]

    # ── Sentence Splitting ────────────────────────────────────────────────────

    def _split_into_sentences(self, text: str) -> list[str]:
        """
        Split raw text into a flat list of atomic sentence strings.

        Two-stage process:
          1. scispaCy's sentence boundary detector splits on full stops and
             other hard boundaries.
          2. :meth:`_split_on_conjunctions` further splits compound sentences
             that join two independent clinical instructions with a conjunction.

        Sentences shorter than four words are discarded as fragments.

        Args:
            text : Raw input text (may contain multiple sentences).

        Returns:
            List of cleaned, atomic sentence strings.
        """
        # Use scispaCy for sentence boundary detection
        doc = self.nlp_sci(text)
        sentences = [sent.text.strip() for sent in doc.sents]
        final = []
        for sentence in sentences:
            final.extend(self._split_on_conjunctions(sentence))
        return [s for s in final if len(s.split()) >= 4]

    def _split_on_conjunctions(self, sentence: str) -> list[str]:
        """
        Split a single sentence on clinical conjunctions from
        :data:`SPLIT_CONJUNCTIONS`.

        Each pattern in the list is applied iteratively so that sentences
        containing multiple conjunctions are split into the correct number of
        parts.

        Args:
            sentence : A single sentence string.

        Returns:
            A list of one or more sub-sentence strings.
        """
        parts = [sentence]
        for pattern in SPLIT_CONJUNCTIONS:
            new_parts = []
            for part in parts:
                split = re.split(pattern, part, flags=re.IGNORECASE)
                new_parts.extend([s.strip() for s in split if s.strip()])
            parts = new_parts
        return parts

    # ── Claim Processing ──────────────────────────────────────────────────────

    def _process_sentence(self, sentence: str, idx: int) -> Optional[Claim]:
        """
        Run the hybrid NLP pipeline on a single sentence and return a
        :class:`Claim`, or ``None`` if the sentence contains no medical content.

        Steps:
          1. Med7 identifies drugs, dosages, and other clinical structure.
          2. scispaCy identifies diseases/conditions not already covered by Med7.
          3. Both entity lists are merged (Med7 is authoritative; scispaCy fills gaps).
          4. Derived fields (drug_names, dosages, conditions) are extracted.
          5. The claim type is classified via keyword scoring.

        Args:
            sentence : A single atomic sentence string.
            idx      : Zero-based position of this sentence in the original text,
                       used to generate a stable ``claim_id``.

        Returns:
            A populated :class:`Claim` object, or ``None`` if the sentence has
            no recognisable medical content.
        """
        # ── Pipeline 1: Med7 — drugs, dosages, clinical structure ─────────────
        doc_med7 = self.nlp_med7(sentence)
        med7_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_med7.ents
        ]

        # ── Pipeline 2: scispaCy — diseases & conditions only ─────────────────
        doc_sci = self.nlp_sci(sentence)
        DISEASE_LABELS = {"DISEASE", "CONDITION", "DISORDER"}
        sci_disease_entities = [
            Entity(
                text=ent.text,
                label=ent.label_,
                start=ent.start_char,
                end=ent.end_char,
            )
            for ent in doc_sci.ents
            if ent.label_ in DISEASE_LABELS
        ]

        # ── Merge: Med7 authoritative; scispaCy fills disease gaps ────────────
        med7_spans    = {(e.start, e.end) for e in med7_entities}
        merged_entities = list(med7_entities)
        for e in sci_disease_entities:
            if (e.start, e.end) not in med7_spans:
                merged_entities.append(e)

        drug_names = self._extract_drugs(med7_entities, sentence)
        dosages    = self._extract_dosages(med7_entities, sentence)
        conditions = self._extract_conditions(sci_disease_entities, sentence)
        claim_type = self._classify_claim_type(sentence, merged_entities)

        # Drop sentences with no medical content at all
        if claim_type == ClaimType.GENERAL_MEDICAL and not merged_entities:
            return None

        requires_verification = claim_type in {
            ClaimType.DOSAGE_CLAIM,
            ClaimType.DRUG_SAFETY_CLAIM,
            ClaimType.DRUG_INTERACTION,
            ClaimType.POPULATION_CLAIM,
        }

        return Claim(
            claim_id=f"claim_{idx}",
            claim_text=sentence,
            claim_type=claim_type,
            entities=merged_entities,
            drug_names=drug_names,
            dosages=dosages,
            conditions=conditions,
            requires_verification=requires_verification,
            confidence=0.0,
        )

    # ── Claim Type Classification ─────────────────────────────────────────────

    def _classify_claim_type(
        self, sentence: str, entities: list[Entity]
    ) -> ClaimType:
        """
        Assign a :class:`ClaimType` to a sentence using keyword frequency scoring.

        Each :data:`CLAIM_TYPE_KEYWORDS` entry is checked against the
        lower-cased sentence.  The type with the most keyword matches wins.
        If no keywords match at all, :attr:`ClaimType.GENERAL_MEDICAL` is returned.

        Args:
            sentence : The raw sentence text.
            entities : Merged entity list (currently unused — reserved for a
                       future entity-boosted scoring pass).

        Returns:
            The best-matching :class:`ClaimType`.
        """
        sentence_lower = sentence.lower()
        scores: dict[ClaimType, int] = {ct: 0 for ct in ClaimType}

        for claim_type, keywords in CLAIM_TYPE_KEYWORDS.items():
            for keyword in keywords:
                if keyword in sentence_lower:
                    scores[claim_type] += 1

        best = max(scores, key=lambda ct: scores[ct])
        return best if scores[best] > 0 else ClaimType.GENERAL_MEDICAL

    # ── Entity Extraction ─────────────────────────────────────────────────────

    @staticmethod
    def _clean_drug_name(text: str) -> str:
        """
        Normalise a raw Med7 drug span by removing common extraction artefacts.

        Artefacts handled:
          - Parenthetical abbreviations appended by the LLM, e.g.
            ``"Fluorometholone (FML)"`` → ``"Fluorometholone"``
          - Leading or trailing punctuation and bracket characters.

        Args:
            text : Raw drug span text as returned by Med7.

        Returns:
            Cleaned drug name string, or an empty string if nothing remains
            after cleaning (caller should discard empty results).
        """
        # Remove parenthetical suffix:  'Drug (ABBR)' → 'Drug'
        text = re.sub(r'\s*\(.*?\)\s*$', '', text)
        # Strip remaining leading/trailing non-alphanumeric characters
        text = re.sub(r'^[^A-Za-z]+|[^A-Za-z0-9]+$', '', text)
        return text.strip()

    def _extract_drugs(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of drug names from Med7 entities and a
        regex suffix fallback.

        Primary source: Med7 ``DRUG`` entities — high-precision clinical NER.
        Fallback: regex matching capitalised tokens whose suffixes are common
        in drug nomenclature (e.g. ``-olone``, ``-mycin``, ``-statin``).

        Args:
            med7_entities : Entities extracted by the Med7 pipeline.
            sentence      : The original sentence text (used for regex fallback).

        Returns:
            Ordered, deduplicated list of drug name strings.
        """
        # Primary: Med7 DRUG label — high precision
        drugs = [
            self._clean_drug_name(e.text)
            for e in med7_entities
            if e.label in MED7_DRUG_LABELS
        ]
        drugs = [d for d in drugs if d]  # drop empty strings after cleaning

        # Secondary: regex suffix fallback (catches brand names Med7 misses)
        regex = re.findall(
            r'\b[A-Z][a-z]+(?:in|ol|ine|ide|ate|mab|nib|pril|sartan|statin|mycin|cillin|zole|olone)\b',
            sentence,
        )
        for d in regex:
            if d not in drugs:
                drugs.append(d)

        return list(dict.fromkeys(drugs))  # deduplicate, preserve order

    def _extract_dosages(
        self, med7_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of dose/strength values from Med7 entities
        and a regex numeric fallback.

        Primary source: Med7 ``DOSAGE`` and ``STRENGTH`` entities.
        Fallback: regex matching numeric values followed by a recognised unit
        symbol (``mg``, ``mcg``, ``ml``, ``g``, ``%``, etc.).

        Args:
            med7_entities : Entities extracted by the Med7 pipeline.
            sentence      : The original sentence text (used for regex fallback).

        Returns:
            Ordered, deduplicated list of dosage strings.
        """
        # Primary: Med7 DOSAGE / STRENGTH labels
        dosages = [e.text for e in med7_entities if e.label in MED7_DOSAGE_LABELS]

        # Secondary: regex — catches formats Med7 might miss
        regex = re.findall(
            r'\b\d+(?:\.\d+)?\s*(?:mg|mcg|ml|g|%|units?|IU|mmol)(?:/(?:day|kg|dose))?\b',
            sentence,
            re.IGNORECASE,
        )
        for d in regex:
            if d.strip() not in dosages:
                dosages.append(d.strip())

        return list(dict.fromkeys(dosages))

    def _extract_conditions(
        self, sci_entities: list[Entity], sentence: str
    ) -> list[str]:
        """
        Build a deduplicated list of disease/condition names from scispaCy
        entities and a known-abbreviation fallback.

        Primary source: scispaCy entities whose label is ``DISEASE``,
        ``CONDITION``, or ``DISORDER`` — the generic ``ENTITY`` label is
        explicitly excluded to suppress noise.

        Fallback: regex matching a fixed vocabulary of common clinical
        abbreviations (e.g. ``CKD``, ``T2DM``, ``UTI``).

        Args:
            sci_entities : Disease-filtered entities from the scispaCy pipeline.
            sentence     : The original sentence text (used for abbreviation
                           fallback).

        Returns:
            Ordered, deduplicated list of condition/disease strings.
        """
        # Primary: scispaCy disease labels only (no ENTITY noise)
        conditions = [e.text for e in sci_entities]

        # Secondary: known clinical abbreviations
        abbrevs = re.findall(
            r'\b(?:CKD|DM|T2DM|HTN|CAD|CHF|COPD|AF|DVT|PE|UTI|PRK|TransPRK)\b',
            sentence,
        )
        for a in abbrevs:
            if a not in conditions:
                conditions.append(a)

        return list(dict.fromkeys(conditions))

    # ── Serialisation ─────────────────────────────────────────────────────────

    def _to_dict(self, claim: Claim) -> dict:
        """
        Serialise a :class:`Claim` to a plain, JSON-serialisable dictionary.

        The ``claim_type`` enum is converted to its string value so the result
        can be passed directly to ``json.dumps`` or a DataFrame constructor.

        Args:
            claim : The :class:`Claim` instance to serialise.

        Returns:
            A ``dict`` with the same keys as the :class:`Claim` dataclass.
        """
        return {
            "claim_id":              claim.claim_id,
            "claim_text":            claim.claim_text,
            "claim_type":            claim.claim_type.value,
            "entities":              [
                {"text": e.text, "label": e.label} for e in claim.entities
            ],
            "drug_names":            claim.drug_names,
            "dosages":               claim.dosages,
            "conditions":            claim.conditions,
            "requires_verification": claim.requires_verification,
            "confidence":            claim.confidence,
        }


## Claim Extractor Test

In [2]:
testdict = ClaimExtractor()

[ClaimExtractor] No GPU detected — running on CPU.
[ClaimExtractor] Loading Med7 model    : en_core_med7_trf


C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\spacy\util.py:910: UserWarning: [W095] Model 'en_core_med7_trf' (3.4.2.1) was trained with spaCy v3.4.2 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\spacy_transformers\layers\hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 't

[ClaimExtractor] Loading scispaCy model: en_core_sci_md


C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


[ClaimExtractor] Hybrid ensemble ready.


In [5]:

test1 = testdict.extract_as_dict(llm_response="To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks. You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.")

[ClaimExtractor] 2 claim(s) extracted.


In [6]:
test1

[{'claim_id': 'claim_0',
  'claim_text': 'To treat corneal haze 5 months post-TransPRK, prescribe Fluorometholone (FML) 0.1% eye drops four times daily for 4 weeks.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Fluorometholone (', 'label': 'DRUG'},
   {'text': '0.1%', 'label': 'STRENGTH'},
   {'text': 'eye', 'label': 'ROUTE'},
   {'text': 'drops', 'label': 'FORM'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 4 weeks', 'label': 'DURATION'}],
  'drug_names': ['Fluorometholone'],
  'dosages': ['0.1%'],
  'conditions': ['TransPRK'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'You may also consider Oral Vitamin C (1000mg/day) to support corneal healing.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Oral', 'label': 'ROUTE'},
   {'text': 'Vitamin C', 'label': 'DRUG'},
   {'text': '1000mg/day', 'label': 'STRENGTH'}],
  'drug_names': ['Vitamin C', 'Vitamin'],
  'dosages': ['1000mg/day'],
 

In [3]:
test2 = testdict.extract_as_dict(llm_response="For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days. If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days. Monitor renal function with creatinine levels weekly.")

C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


[ClaimExtractor] 3 claim(s) extracted.


In [4]:
test2

[{'claim_id': 'claim_0',
  'claim_text': 'For a patient with a confirmed UTI, prescribe Trimethoprim 200mg twice daily for 7 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Trimethoprim', 'label': 'DRUG'},
   {'text': '200mg', 'label': 'STRENGTH'},
   {'text': 'twice daily', 'label': 'FREQUENCY'},
   {'text': 'for 7 days', 'label': 'DURATION'}],
  'drug_names': ['Trimethoprim'],
  'dosages': ['200mg'],
  'conditions': ['UTI'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'If symptoms persist after 48 hours, switch to Nitrofurantoin 100mg four times daily for 5 days.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Nitrofurantoin', 'label': 'DRUG'},
   {'text': '100mg', 'label': 'STRENGTH'},
   {'text': 'four times daily', 'label': 'FREQUENCY'},
   {'text': 'for 5 days', 'label': 'DURATION'}],
  'drug_names': ['Nitrofurantoin'],
  'dosages': ['100mg'],
  'conditions': [],
  'requires_verification': True,
  'co

In [3]:
test3 = testdict.extract_as_dict(llm_response="In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis. Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population. Monitor HbA1c every 3 months.")

C:\Users\basim\PycharmProjects\Clinical-AI-Safety-Monitor\.venv\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):


[ClaimExtractor] 3 claim(s) extracted.


In [4]:
test3

[{'claim_id': 'claim_0',
  'claim_text': 'In a T2DM patient with CKD stage 3, avoid Metformin 500mg if eGFR drops below 30 ml/min due to risk of lactic acidosis.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Metformin', 'label': 'DRUG'},
   {'text': '500mg', 'label': 'STRENGTH'}],
  'drug_names': ['Metformin'],
  'dosages': ['500mg', '30 ml'],
  'conditions': ['T2DM', 'CKD'],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_1',
  'claim_text': 'Consider switching to Sitagliptin 50mg once daily, which is renally dosed and safer in this population.',
  'claim_type': 'DOSAGE_CLAIM',
  'entities': [{'text': 'Sitagliptin', 'label': 'DRUG'},
   {'text': '50mg', 'label': 'STRENGTH'},
   {'text': 'once daily', 'label': 'FREQUENCY'}],
  'drug_names': ['Sitagliptin'],
  'dosages': ['50mg'],
  'conditions': [],
  'requires_verification': True,
  'confidence': 0.0},
 {'claim_id': 'claim_2',
  'claim_text': 'Monitor HbA1c every 3 months.',
  'claim_type': 'PR

## Knowledge Verifier